# Voxa V3 Pipeline Evaluation

This notebook evaluates 5 configurations of the Voxa voice authentication pipeline on the **Google Speech Commands v2 dataset**.

## V3 Improvements Under Test
- **CMN-Only Normalization**: Cepstral Mean Normalization without variance normalization, preserving discriminative variance information.
- **Feature Warping**: Sliding-window CDF-based warping to Gaussianize features for channel robustness.
- **RASTA Filtering**: Bandpass temporal modulation filter to suppress slow channel effects and fast noise.
- **Fine-Grained Threshold Sweep**: Denser grid search (0.25 abs step, 0.05 margin step) for tighter operating points.
- **top_k=3 Consensus**: Averaging top-3 template distances instead of top-2 for more stable matching.

## Configurations
| Config | Description |
| --- | --- |
| 1 | **Baseline** — CMVN, no AGC, no banding, top_k=2, coarse thresholds |
| 2 | **Banded+AGC** — CMVN, AGC, adaptive banding, top_k=2, coarse thresholds |
| 3 | **CMN+Fine** — CMN-only, AGC, adaptive banding, top_k=3, fine thresholds |
| 4 | **CMN+FeatWarp** — CMN-only, AGC, adaptive banding, top_k=3, fine thresholds, feature warping |
| 5 | **CMN+RASTA** — CMN-only, AGC, adaptive banding, top_k=3, fine thresholds, RASTA filter |

In [1]:
# Install webrtcvad and tabulate for formatting
!pip install -q webrtcvad tabulate pandas numpy scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
import urllib.request
import tarfile
import time
import random
import numpy as np
import scipy.io.wavfile as wav
from scipy.fftpack import dct
from pathlib import Path
from typing import List, Dict, Tuple
import pandas as pd
from tabulate import tabulate

print("Imports successful.")

Imports successful.


In [3]:
# Dataset downloader and verification
possible_paths = [
    "/kaggle/input/datasets/sylkaladin/speech-commands-v2",
    "/kaggle/input/google-speech-commands-v2",
    "/kaggle/input/datasets/sylkaladin/speech-commands-v2",
    "/kaggle/input/speech-commands-v2/speech_commands_v2",
    "./speech_commands_v2"
]

DATASET_PATH = None
for p in possible_paths:
    if os.path.exists(p) and any(os.path.isdir(os.path.join(p, d)) for d in ["yes", "no", "up", "down"]):
        DATASET_PATH = p
        print(f"Found dataset at: {DATASET_PATH}")
        break

if DATASET_PATH is None:
    print("Dataset not found in Kaggle inputs. Downloading dataset...")
    url = "http://download.tensorflow.org/data/speech_commands_v2.tar.gz"
    dest = "./speech_commands_v2.tar.gz"
    urllib.request.urlretrieve(url, dest)
    print("Extracting dataset...")
    with tarfile.open(dest, "r:gz") as tar:
        tar.extractall("./speech_commands_v2")
    DATASET_PATH = "./speech_commands_v2"
    print(f"Dataset extracted to: {DATASET_PATH}")

Found dataset at: /kaggle/input/datasets/sylkaladin/speech-commands-v2


In [4]:
# Phase 1: WebRTC VAD
import webrtcvad

class VADConfig:
    def __init__(self, sample_rate=16000, frame_ms=20, aggressiveness=2,
                 speech_trigger_frames=8, silence_boundary_frames=15,
                 min_segment_ms=400, max_segment_ms=2000):
        self.sample_rate = sample_rate
        self.frame_ms = frame_ms
        self.frame_size = int(sample_rate * frame_ms / 1000)
        self.aggressiveness = aggressiveness
        self.M = speech_trigger_frames
        self.N = silence_boundary_frames
        self.min_samples = int(sample_rate * min_segment_ms / 1000)
        self.max_samples = int(sample_rate * max_segment_ms / 1000)

class VoxaVAD:
    def __init__(self, config=None):
        self.config = config if config else VADConfig()
        self.vad = webrtcvad.Vad(self.config.aggressiveness)
        self.reset()
        
    def reset(self):
        self.state = "SILENCE"
        self.speech_frame_count = 0
        self.silence_frame_count = 0
        self.segment_buffer = []
        
    def process_frame(self, frame_bytes: bytes) -> tuple:
        is_speech = self.vad.is_speech(frame_bytes, self.config.sample_rate)
        frame_samples = np.frombuffer(frame_bytes, dtype=np.int16)
        
        if self.state == "SILENCE":
            if is_speech:
                self.speech_frame_count += 1
                self.segment_buffer.append(frame_samples)
                if self.speech_frame_count >= self.config.M:
                    self.state = "SPEECH_COLLECTING"
                    self.silence_frame_count = 0
            else:
                self.speech_frame_count = 0
                self.segment_buffer = []
                
        elif self.state == "SPEECH_COLLECTING":
            self.segment_buffer.append(frame_samples)
            if not is_speech:
                self.silence_frame_count += 1
                if self.silence_frame_count >= self.config.N:
                    segment = np.concatenate(self.segment_buffer)
                    self.reset()
                    if self.config.min_samples <= len(segment) <= self.config.max_samples:
                        return ("SEGMENT_COMPLETE", segment)
                    else:
                        return ("DISCARDED", None)
            else:
                self.silence_frame_count = 0
                total_samples = sum(len(b) for b in self.segment_buffer)
                if total_samples > self.config.max_samples:
                    self.reset()
                    return ("DISCARDED", None)
                    
        return (self.state, None)
        
    def process_audio(self, pcm_int16: np.ndarray) -> list:
        segments = []
        frame_bytes_size = self.config.frame_size * 2
        audio_bytes = pcm_int16.tobytes()
        
        for i in range(0, len(audio_bytes) - frame_bytes_size + 1, frame_bytes_size):
            frame = audio_bytes[i:i + frame_bytes_size]
            state, segment = self.process_frame(frame)
            if segment is not None:
                segments.append(segment)
                
        # Trailing buffer check
        if self.state == "SPEECH_COLLECTING" and len(self.segment_buffer) > 0:
            segment = np.concatenate(self.segment_buffer)
            if self.config.min_samples <= len(segment) <= self.config.max_samples:
                segments.append(segment)
        self.reset()
        return segments

In [5]:
# Phase 2: MFCC Feature Extractor
def apply_agc(signal: np.ndarray, target_rms=0.05, max_gain=10.0) -> np.ndarray:
    rms = np.sqrt(np.mean(signal ** 2) + 1e-10)
    if rms < 1e-5:
        return signal
    scale = target_rms / rms
    scale = min(scale, max_gain)
    scaled_signal = signal * scale
    return np.clip(scaled_signal, -1.0, 1.0)

def wiener_filter(pcm_int16: np.ndarray, sr=16000, n_fft=512, hop_len=128, noise_frames=8) -> np.ndarray:
    if len(pcm_int16) < n_fft:
        return pcm_int16
    signal = pcm_int16.astype(np.float64) / 32768.0
    frame_len = n_fft
    n_frames = 1 + (len(signal) - frame_len) // hop_len
    if n_frames <= 0:
        return pcm_int16
    # Manual Hann window
    window = 0.5 * (1.0 - np.cos(2.0 * np.pi * np.arange(frame_len) / (frame_len - 1)))
    frames = np.zeros((n_frames, frame_len))
    for i in range(n_frames):
        start = i * hop_len
        frames[i] = signal[start:start + frame_len] * window
    spec = np.fft.rfft(frames, n=n_fft)
    power = np.abs(spec) ** 2
    noise_est = np.mean(power[:min(noise_frames, n_frames)], axis=0)
    noise_est = np.maximum(noise_est, 1e-10)
    beta = 0.02  # spectral floor
    gain = (power - noise_est) / (power + 1e-10)
    gain = np.clip(gain, beta, 1.0)
    clean_spec = spec * gain
    clean_frames = np.fft.irfft(clean_spec, n=n_fft)
    recon_len = (n_frames - 1) * hop_len + frame_len
    recon_signal = np.zeros(recon_len)
    window_sum = np.zeros(recon_len)
    for i in range(n_frames):
        start = i * hop_len
        recon_signal[start:start + frame_len] += clean_frames[i]
        window_sum[start:start + frame_len] += window
    recon_signal = recon_signal / (window_sum + 1e-10)
    if len(recon_signal) < len(pcm_int16):
        padded = np.zeros(len(pcm_int16))
        padded[:len(recon_signal)] = recon_signal
        recon_signal = padded
    elif len(recon_signal) > len(pcm_int16):
        recon_signal = recon_signal[:len(pcm_int16)]
    return np.clip(recon_signal * 32768.0, -32768, 32767).astype(np.int16)

def pre_emphasis(signal, coeff=0.97):
    return np.append(signal[0], signal[1:] - coeff * signal[:-1])

def mel_to_hz(mel):
    return 700 * (10**(mel / 2595.0) - 1)

def hz_to_mel(hz):
    return 2595 * np.log10(1 + hz / 700.0)

def create_mel_filterbank(n_filters, n_fft, sample_rate, f_min, f_max):
    mel_min = hz_to_mel(f_min)
    mel_max = hz_to_mel(f_max)
    mel_points = np.linspace(mel_min, mel_max, n_filters + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)
    filterbank = np.zeros((n_filters, n_fft // 2 + 1))
    for i in range(n_filters):
        for j in range(bins[i], bins[i+1]):
            filterbank[i, j] = (j - bins[i]) / (bins[i+1] - bins[i])
        for j in range(bins[i+1], bins[i+2]):
            filterbank[i, j] = (bins[i+2] - j) / (bins[i+2] - bins[i+1])
    return filterbank

def compute_deltas(features, K=2):
    T, D = features.shape
    deltas = np.zeros_like(features)
    denominator = 2 * sum(n**2 for n in range(1, K + 1))
    for t in range(T):
        for n in range(1, K + 1):
            t_plus = min(t + n, T - 1)
            t_minus = max(t - n, 0)
            deltas[t] += n * (features[t_plus] - features[t_minus])
        deltas[t] /= denominator
    return deltas

def feature_warping(features, window=301):
    T, D = features.shape
    if T < 3:
        return features
    warped = np.zeros_like(features)
    half_w = window // 2
    def inv_normal_cdf(p):
        p = np.clip(p, 1e-6, 1.0 - 1e-6)
        t = np.where(p < 0.5, p, 1.0 - p)
        t = np.sqrt(-2.0 * np.log(t))
        c0, c1, c2 = 2.515517, 0.802853, 0.010328
        d1, d2, d3 = 1.432788, 0.189269, 0.001308
        result = t - (c0 + c1 * t + c2 * t * t) / (1.0 + d1 * t + d2 * t * t + d3 * t * t * t)
        return np.where(p < 0.5, -result, result)
    for d in range(D):
        col = features[:, d]
        for t in range(T):
            start = max(0, t - half_w)
            end = min(T, t + half_w + 1)
            local = col[start:end]
            n_local = len(local)
            rank = np.sum(local < col[t]) + 0.5 * np.sum(local == col[t])
            quantile = (rank + 0.5) / n_local
            warped[t, d] = inv_normal_cdf(quantile)
    return warped

def rasta_filter(features):
    T, D = features.shape
    if T < 5:
        return features
    numer = np.array([0.2, 0.1, 0.0, -0.1, -0.2])
    pole = 0.98
    filtered = np.zeros_like(features)
    for d in range(D):
        x = features[:, d]
        fir_out = np.convolve(x, numer, mode='same')
        y = np.zeros(T)
        y[0] = fir_out[0]
        for t in range(1, T):
            y[t] = fir_out[t] + pole * y[t - 1]
        filtered[:, d] = y
    return filtered

def extract_mfcc(pcm_int16: np.ndarray, config) -> np.ndarray:
    sr = config["sample_rate"]
    frame_len = int(sr * config["frame_size_ms"] / 1000)
    hop_len = int(sr * config["hop_size_ms"] / 1000)
    n_fft = config["n_fft"]
    signal = pcm_int16.astype(np.float64) / 32768.0
    if config.get("agc", False):
        signal = apply_agc(signal, target_rms=config.get("agc_target_rms", 0.05))
    signal = pre_emphasis(signal, config["pre_emphasis"])
    n_frames = 1 + (len(signal) - frame_len) // hop_len
    if n_frames <= 0:
        return np.zeros((0, 40))
    frames = np.zeros((n_frames, frame_len))
    for i in range(n_frames):
        start = i * hop_len
        frames[i] = signal[start:start + frame_len]
    if config["window"] == "hamming":
        frames *= np.hamming(frame_len)
    mag = np.abs(np.fft.rfft(frames, n=n_fft))
    power = mag ** 2 / n_fft
    filterbank = create_mel_filterbank(config["n_mels"], n_fft, sr, config["f_min"], config["f_max"])
    mel_spec = np.dot(power, filterbank.T)
    mel_spec = np.maximum(mel_spec, 1e-10)
    log_mel = np.log(mel_spec)
    energy = np.log(np.sum(power, axis=1) + 1e-10).reshape(-1, 1)
    mfcc = dct(log_mel, type=2, axis=1, norm='ortho')[:, :config["n_mfcc"]]
    delta = compute_deltas(mfcc, config["delta_window_K"])
    delta_delta = compute_deltas(delta, config["delta_window_K"])
    features = np.hstack([mfcc, delta, delta_delta, energy])
    cmvn_mode = config.get("cmvn", "per_utterance")
    if cmvn_mode == "per_utterance":
        features = (features - np.mean(features, axis=0)) / (np.std(features, axis=0) + 1e-10)
    elif cmvn_mode == "cmn_only":
        features = features - np.mean(features, axis=0)
    if config.get("feature_warping", False):
        warp_window = config.get("feature_warping_window", 301)
        features = feature_warping(features, window=warp_window)
    if config.get("rasta_filter", False):
        features = rasta_filter(features)
    return features

In [6]:
# Phase 3: DTW Matcher
def dtw_distance(X: np.ndarray, Y: np.ndarray, band_width=None) -> float:
    N, M = len(X), len(Y)
    if N == 0 or M == 0:
        return np.inf
        
    if band_width == "adaptive":
        band_width = max(15, abs(N - M) + 5)
        
    D = np.full((N + 1, M + 1), np.inf)
    D[0, 0] = 0.0
    for i in range(1, N + 1):
        j_start = 1
        j_end = M
        if band_width is not None:
            j_start = max(1, i - band_width)
            j_end = min(M, i + band_width)
        for j in range(j_start, j_end + 1):
            cost = np.sqrt(np.sum((X[i-1] - Y[j-1]) ** 2))
            D[i, j] = cost + min(D[i-1, j], D[i, j-1], D[i-1, j-1])
    if D[N, M] == np.inf:
        return np.inf
    path_length = 0
    i, j = N, M
    while i > 0 or j > 0:
        path_length += 1
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            argmin = np.argmin([D[i-1, j-1], D[i-1, j], D[i, j-1]])
            if argmin == 0:
                i, j = i-1, j-1
            elif argmin == 1:
                i -= 1
            else:
                j -= 1
    return float(D[N, M] / max(path_length, 1))

def consensus_match(test_features: np.ndarray, intent_templates: dict, top_k=2, band_width=None) -> list:
    results = []
    for intent_name, templates in intent_templates.items():
        if not templates:
            results.append((intent_name, np.inf))
            continue
        distances = []
        for template in templates:
            d = dtw_distance(template, test_features, band_width)
            distances.append(d)
        distances.sort()
        k_to_use = min(top_k, len(distances))
        avg_top_k = np.mean(distances[:k_to_use]) if k_to_use > 0 else np.inf
        results.append((intent_name, avg_top_k))
    results.sort(key=lambda x: x[1])
    return results

In [7]:
# Dataset Loader Utilities
def load_wav_pcm(filepath):
    sr, y = wav.read(filepath)
    if len(y.shape) > 1:
        y = y.mean(axis=1)
    return y.astype(np.int16)

def parse_dataset(data_dir):
    path = Path(data_dir)
    records = []
    target_words = {"yes", "no", "up", "down", "left", "right"}
    unknown_words = {"one", "two", "three", "four", "go", "stop"}
    all_words = target_words.union(unknown_words)
    print("Scanning directory... ")
    for word_dir in path.iterdir():
        if not word_dir.is_dir() or word_dir.name.startswith('_'):
            continue
        word = word_dir.name.lower()
        if word not in all_words:
            continue
        for fpath in word_dir.glob("*.wav"):
            parts = fpath.stem.split('_')
            if len(parts) >= 2:
                speaker_id = parts[0]
                records.append({
                    "word": word,
                    "speaker_id": speaker_id,
                    "filepath": str(fpath),
                    "is_target": word in target_words
                })
    print(f"Found {len(records)} total records for target/unknown words.")
    return records

In [8]:
# Setup dataset
records = parse_dataset(DATASET_PATH)
df = pd.DataFrame(records)

qualified_speakers = []
for spk_id, group in df.groupby("speaker_id"):
    word_counts = group[group["is_target"]]["word"].value_counts()
    unknown_count = len(group[~group["is_target"]])
    if len(word_counts) == 6 and (word_counts >= 6).all() and unknown_count >= 5:
        qualified_speakers.append(spk_id)
print(f"Number of qualified speakers: {len(qualified_speakers)}")

# Limit to top 7 speakers to keep runtime fast
eval_speakers = qualified_speakers[:7]
print(f"Evaluating on 7 speakers: {eval_speakers}")

# Pre-cache noise files
all_wav_files = [str(p) for p in Path(DATASET_PATH).rglob("*.wav")]
SPEECH_FILES = [f for f in all_wav_files if "_background_noise_" not in f]
ENV_NOISE_FILES = [f for f in all_wav_files if "_background_noise_" in f]
print(f"Pre-cached {len(SPEECH_FILES)} speech files and {len(ENV_NOISE_FILES)} noise files.")

ENV_NOISE_DATA = []
for fp in ENV_NOISE_FILES:
    try:
        sr, y = wav.read(fp)
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        if len(y) > 0:
            ENV_NOISE_DATA.append(y.astype(np.int16))
    except Exception as e:
        print(f"Warning: failed to load background noise {fp}: {e}")

print("Pre-generating speech babble noise source...")
babble_duration = 16000 * 20
BABBLE_NOISE_SOURCE = np.zeros(babble_duration, dtype=np.float64)
if len(SPEECH_FILES) >= 15:
    selected = random.sample(SPEECH_FILES, 15)
    for fp in selected:
        try:
            sr, y = wav.read(fp)
            if len(y.shape) > 1:
                y = y.mean(axis=1)
            if len(y) >= babble_duration:
                segment = y[:babble_duration]
            else:
                repeats = (babble_duration // len(y)) + 1
                segment = np.tile(y, repeats)[:babble_duration]
            BABBLE_NOISE_SOURCE += segment.astype(np.float64)
        except Exception:
            continue
    BABBLE_NOISE_SOURCE /= len(selected)
BABBLE_NOISE_SOURCE = BABBLE_NOISE_SOURCE.astype(np.int16)
print("Noise pre-caching and pre-generation completed.")

Scanning directory... 
Found 46181 total records for target/unknown words.
Number of qualified speakers: 7
Evaluating on 7 speakers: ['5f9cd2eb', '893705bb', 'b5cf6ea8', 'b66f4f93', 'c1d39ce8', 'cce7416f', 'ddedba85']
Pre-cached 105829 speech files and 6 noise files.


/tmp/ipykernel_17/62732946.py:26: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, y = wav.read(fp)


Pre-generating speech babble noise source...
Noise pre-caching and pre-generation completed.


In [9]:
# Mixing Helpers
def get_noise_segment(noise_source, target_len):
    if len(noise_source) >= target_len:
        start = random.randint(0, len(noise_source) - target_len)
        return noise_source[start:start + target_len]
    else:
        repeats = (target_len // len(noise_source)) + 1
        tiled = np.tile(noise_source, repeats)
        return tiled[:target_len]

def add_noise(signal, noise, snr_db):
    signal_energy = np.mean(signal.astype(np.float64) ** 2)
    noise_energy = np.mean(noise.astype(np.float64) ** 2)
    if noise_energy == 0:
        return signal
    target_noise_energy = signal_energy / (10 ** (snr_db / 10.0))
    scale = np.sqrt(target_noise_energy / noise_energy)
    noisy_signal = signal.astype(np.float64) + scale * noise.astype(np.float64)
    noisy_signal = np.clip(noisy_signal, -32768, 32767)
    return noisy_signal.astype(np.int16)

def load_environmental_noise():
    if not ENV_NOISE_DATA:
        return np.zeros(16000, dtype=np.int16)
    return random.choice(ENV_NOISE_DATA)

def generate_babble_noise(duration_samples=16000):
    return get_noise_segment(BABBLE_NOISE_SOURCE, duration_samples)

In [10]:
# Full Evaluation runner for a specific configuration
def run_configuration_evaluation(config_name, mfcc_cfg, dtw_cfg, use_wiener_pre_vad=False, top_k=2, fine_thresholds=False):
    print(f"\n=========================================================")
    print(f"  RUNNING EVALUATION: {config_name}")
    print(f"=========================================================")
    
    vad_config = VADConfig(aggressiveness=2)
    enrolled_templates = {}
    test_samples_by_speaker = {}
    
    # 1. Enroll templates and prepare test samples
    for spk in eval_speakers:
        spk_df = df[df["speaker_id"] == spk]
        intent_templates = {}
        test_samples = []
        
        for word in ["yes", "no", "up", "down", "left", "right"]:
            word_df = spk_df[spk_df["word"] == word]
            filepaths = word_df["filepath"].tolist()
            enroll_files = filepaths[:5]
            test_files = filepaths[5:]
            
            templates = []
            for fp in enroll_files:
                audio = load_wav_pcm(fp)
                if use_wiener_pre_vad:
                    audio = wiener_filter(audio)
                vad = VoxaVAD(vad_config)
                segments = vad.process_audio(audio)
                if segments:
                    seg = max(segments, key=len)
                    feat = extract_mfcc(seg, mfcc_cfg)
                    if len(feat) > 0:
                        templates.append(feat)
                else:
                    feat = extract_mfcc(audio, mfcc_cfg)
                    if len(feat) > 0:
                        templates.append(feat)
            intent_templates[word] = templates
            
            for fp in test_files:
                test_samples.append({
                    "filepath": fp,
                    "true_intent": word,
                    "is_target": True
                })
                
        # Add unknown test samples
        unknown_df = spk_df[~spk_df["is_target"]]
        for fp in unknown_df["filepath"].tolist():
            test_samples.append({
                "filepath": fp,
                "true_intent": "unknown",
                "is_target": False
            })
            
        enrolled_templates[spk] = intent_templates
        test_samples_by_speaker[spk] = test_samples
        
    # 2. Sweep thresholds on Clean to find optimal operating point
    clean_results = []
    for spk in eval_speakers:
        templates = enrolled_templates[spk]
        samples = test_samples_by_speaker[spk]
        
        for sample in samples:
            audio = load_wav_pcm(sample["filepath"])
            t0 = time.perf_counter()
            if use_wiener_pre_vad:
                audio = wiener_filter(audio)
            vad = VoxaVAD(vad_config)
            segments = vad.process_audio(audio)
            
            if not segments:
                latency = (time.perf_counter() - t0) * 1000
                clean_results.append({
                    "true_intent": sample["true_intent"],
                    "predicted_intent": "unknown",
                    "vad_triggered": False,
                    "latency_ms": latency,
                    "best_distance": np.inf,
                    "margin": np.inf
                })
                continue
                
            seg = max(segments, key=len)
            feat = extract_mfcc(seg, mfcc_cfg)
            if len(feat) == 0:
                latency = (time.perf_counter() - t0) * 1000
                clean_results.append({
                    "true_intent": sample["true_intent"],
                    "predicted_intent": "unknown",
                    "vad_triggered": True,
                    "latency_ms": latency,
                    "best_distance": np.inf,
                    "margin": np.inf
                })
                continue
                
            match_scores = consensus_match(feat, templates, top_k=top_k, band_width=dtw_cfg["sakoe_chiba_band"])
            latency = (time.perf_counter() - t0) * 1000
            
            if len(match_scores) >= 2:
                best_intent, best_dist = match_scores[0]
                second_dist = match_scores[1][1]
                margin = second_dist - best_dist
            else:
                best_intent, best_dist = "unknown", np.inf
                margin = 0.0
                
            clean_results.append({
                "true_intent": sample["true_intent"],
                "predicted_intent": best_intent,
                "best_distance": best_dist,
                "margin": margin,
                "vad_triggered": True,
                "latency_ms": latency
            })
            
    df_clean = pd.DataFrame(clean_results)
    avg_latency = df_clean[df_clean["vad_triggered"]]["latency_ms"].mean()
    
    # Grid search for optimal clean thresholds
    best_tar, best_far, best_urr = 0.0, 1.0, 0.0
    best_theta_abs, best_theta_margin = 0.0, 0.0
    
    if fine_thresholds:
        abs_sweep = np.arange(3.0, 12.0, 0.25)
        margin_sweep = np.arange(0.0, 3.0, 0.05)
    else:
        abs_sweep = np.linspace(3.0, 15.0, 25)
        margin_sweep = np.linspace(0.0, 3.0, 20)
    
    for theta_abs in abs_sweep:
        for theta_margin in margin_sweep:
            tp, fp, tn, fn = 0, 0, 0, 0
            for idx, row in df_clean.iterrows():
                if not row["vad_triggered"] or row["best_distance"] == np.inf:
                    pred = "unknown"
                else:
                    if row["best_distance"] < theta_abs and row["margin"] > theta_margin:
                        pred = row["predicted_intent"]
                    else:
                        pred = "unknown"
                true = row["true_intent"]
                if true == "unknown":
                    if pred == "unknown": tn += 1
                    else: fp += 1
                else:
                    if pred == true: tp += 1
                    else: fn += 1
            tar = tp / (tp + fn) if (tp + fn) > 0 else 0
            far = fp / (fp + tn) if (fp + tn) > 0 else 0
            urr = tn / (tn + fp) if (tn + fp) > 0 else 0
            if far <= 0.03 and tar >= best_tar:
                best_tar = tar
                best_far = far
                best_urr = urr
                best_theta_abs = theta_abs
                best_theta_margin = theta_margin
                
    print(f"Optimized clean thresholds: theta_abs = {best_theta_abs:.2f}, theta_margin = {best_theta_margin:.2f}")
    print(f"Clean results: TAR = {best_tar*100:.1f}%, FAR = {best_far*100:.1f}%, URR = {best_urr*100:.1f}%, Latency = {avg_latency:.1f}ms")
    
    # 3. Run Noisy Evaluations
    scenarios = [
        ("Clean (0dB Noise)", "none", 999),
        ("20dB Environmental Noise", "environmental", 20),
        ("20dB Speech Babble Noise", "babble", 20),
        ("10dB Environmental Noise", "environmental", 10),
        ("10dB Speech Babble Noise", "babble", 10),
        ("5dB Environmental Noise", "environmental", 5),
        ("5dB Speech Babble Noise", "babble", 5)
    ]
    
    scen_results = {}
    
    for label, noise_type, snr in scenarios:
        if noise_type == "none":
            scen_results[label] = (best_tar, best_far, best_urr)
            continue
            
        run_log = []
        for spk in eval_speakers:
            templates = enrolled_templates[spk]
            samples = test_samples_by_speaker[spk]
            
            for sample in samples:
                audio = load_wav_pcm(sample["filepath"])
                if noise_type == "environmental":
                    noise = get_noise_segment(load_environmental_noise(), len(audio))
                elif noise_type == "babble":
                    noise = generate_babble_noise(len(audio))
                else:
                    noise = np.zeros_like(audio)
                noisy_audio = add_noise(audio, noise, snr)
                
                if use_wiener_pre_vad:
                    noisy_audio = wiener_filter(noisy_audio)
                    
                vad = VoxaVAD(vad_config)
                segments = vad.process_audio(noisy_audio)
                
                if not segments:
                    run_log.append({
                        "true_intent": sample["true_intent"],
                        "predicted_intent": "unknown",
                        "vad_triggered": False,
                        "best_distance": np.inf,
                        "margin": np.inf
                    })
                    continue
                    
                seg = max(segments, key=len)
                feat = extract_mfcc(seg, mfcc_cfg)
                if len(feat) == 0:
                    run_log.append({
                        "true_intent": sample["true_intent"],
                        "predicted_intent": "unknown",
                        "vad_triggered": True,
                        "best_distance": np.inf,
                        "margin": np.inf
                    })
                    continue
                    
                match_scores = consensus_match(feat, templates, top_k=top_k, band_width=dtw_cfg["sakoe_chiba_band"])
                
                if len(match_scores) >= 2:
                    best_intent, best_dist = match_scores[0]
                    second_dist = match_scores[1][1]
                    margin = second_dist - best_dist
                else:
                    best_intent, best_dist = "unknown", np.inf
                    margin = 0.0
                    
                run_log.append({
                    "true_intent": sample["true_intent"],
                    "predicted_intent": best_intent,
                    "best_distance": best_dist,
                    "margin": margin,
                    "vad_triggered": True
                })
                
        df_run = pd.DataFrame(run_log)
        tp, fp, tn, fn = 0, 0, 0, 0
        for idx, row in df_run.iterrows():
            if not row["vad_triggered"] or row["best_distance"] == np.inf:
                pred = "unknown"
            else:
                if row["best_distance"] < best_theta_abs and row["margin"] > best_theta_margin:
                    pred = row["predicted_intent"]
                else:
                    pred = "unknown"
            true = row["true_intent"]
            if true == "unknown":
                if pred == "unknown": tn += 1
                else: fp += 1
            else:
                if pred == true: tp += 1
                else: fn += 1
        tar = tp / (tp + fn) if (tp + fn) > 0 else 0
        far = fp / (fp + tn) if (fp + tn) > 0 else 0
        urr = tn / (tn + fp) if (tn + fp) > 0 else 0
        scen_results[label] = (tar, far, urr)
        
    return scen_results, avg_latency

In [11]:
# Configuration 1: Baseline (No AGC, No Banding, CMVN, top_k=2, coarse thresholds)
mfcc_cfg_1 = {
    "sample_rate": 16000, "frame_size_ms": 25, "hop_size_ms": 10,
    "n_fft": 512, "n_mels": 40, "n_mfcc": 13, "f_min": 100, "f_max": 8000,
    "window": "hamming", "pre_emphasis": 0.97, "cmvn": "per_utterance", "delta_window_K": 2,
    "agc": False
}
dtw_cfg_1 = {"sakoe_chiba_band": None}

cfg1_res, cfg1_lat = run_configuration_evaluation(
    "Config 1: Baseline (No enhancements)", mfcc_cfg_1, dtw_cfg_1,
    use_wiener_pre_vad=False, top_k=2, fine_thresholds=False
)

# Configuration 2: Banded DTW + AGC (CMVN, top_k=2, coarse thresholds)
mfcc_cfg_2 = {
    "sample_rate": 16000, "frame_size_ms": 25, "hop_size_ms": 10,
    "n_fft": 512, "n_mels": 40, "n_mfcc": 13, "f_min": 100, "f_max": 8000,
    "window": "hamming", "pre_emphasis": 0.97, "cmvn": "per_utterance", "delta_window_K": 2,
    "agc": True, "agc_target_rms": 0.05
}
dtw_cfg_2 = {"sakoe_chiba_band": "adaptive"}

cfg2_res, cfg2_lat = run_configuration_evaluation(
    "Config 2: Banded DTW + AGC", mfcc_cfg_2, dtw_cfg_2,
    use_wiener_pre_vad=False, top_k=2, fine_thresholds=False
)

# Configuration 3: CMN-Only + top_k=3 + Fine Thresholds
mfcc_cfg_3 = {
    "sample_rate": 16000, "frame_size_ms": 25, "hop_size_ms": 10,
    "n_fft": 512, "n_mels": 40, "n_mfcc": 13, "f_min": 100, "f_max": 8000,
    "window": "hamming", "pre_emphasis": 0.97, "cmvn": "cmn_only", "delta_window_K": 2,
    "agc": True, "agc_target_rms": 0.05
}
dtw_cfg_3 = {"sakoe_chiba_band": "adaptive"}

cfg3_res, cfg3_lat = run_configuration_evaluation(
    "Config 3: CMN-Only + top_k=3 + Fine Thresholds", mfcc_cfg_3, dtw_cfg_3,
    use_wiener_pre_vad=False, top_k=3, fine_thresholds=True
)

# Configuration 4: CMN + Feature Warping + top_k=3
mfcc_cfg_4 = {
    "sample_rate": 16000, "frame_size_ms": 25, "hop_size_ms": 10,
    "n_fft": 512, "n_mels": 40, "n_mfcc": 13, "f_min": 100, "f_max": 8000,
    "window": "hamming", "pre_emphasis": 0.97, "cmvn": "cmn_only", "delta_window_K": 2,
    "agc": True, "agc_target_rms": 0.05,
    "feature_warping": True, "feature_warping_window": 301
}
dtw_cfg_4 = {"sakoe_chiba_band": "adaptive"}

cfg4_res, cfg4_lat = run_configuration_evaluation(
    "Config 4: CMN + Feature Warping + top_k=3", mfcc_cfg_4, dtw_cfg_4,
    use_wiener_pre_vad=False, top_k=3, fine_thresholds=True
)

# Configuration 5: CMN + RASTA + top_k=3
mfcc_cfg_5 = {
    "sample_rate": 16000, "frame_size_ms": 25, "hop_size_ms": 10,
    "n_fft": 512, "n_mels": 40, "n_mfcc": 13, "f_min": 100, "f_max": 8000,
    "window": "hamming", "pre_emphasis": 0.97, "cmvn": "cmn_only", "delta_window_K": 2,
    "agc": True, "agc_target_rms": 0.05,
    "rasta_filter": True
}
dtw_cfg_5 = {"sakoe_chiba_band": "adaptive"}

cfg5_res, cfg5_lat = run_configuration_evaluation(
    "Config 5: CMN + RASTA + top_k=3", mfcc_cfg_5, dtw_cfg_5,
    use_wiener_pre_vad=False, top_k=3, fine_thresholds=True
)


  RUNNING EVALUATION: Config 1: Baseline (No enhancements)
Optimized clean thresholds: theta_abs = 6.50, theta_margin = 0.32
Clean results: TAR = 80.7%, FAR = 2.9%, URR = 97.1%, Latency = 1018.3ms

  RUNNING EVALUATION: Config 2: Banded DTW + AGC
Optimized clean thresholds: theta_abs = 6.50, theta_margin = 0.32
Clean results: TAR = 81.3%, FAR = 2.9%, URR = 97.1%, Latency = 527.4ms

  RUNNING EVALUATION: Config 3: CMN-Only + top_k=3 + Fine Thresholds
Optimized clean thresholds: theta_abs = 9.50, theta_margin = 2.00
Clean results: TAR = 59.4%, FAR = 2.6%, URR = 97.4%, Latency = 510.7ms

  RUNNING EVALUATION: Config 4: CMN + Feature Warping + top_k=3
Optimized clean thresholds: theta_abs = 7.50, theta_margin = 0.45
Clean results: TAR = 75.4%, FAR = 2.6%, URR = 97.4%, Latency = 567.8ms

  RUNNING EVALUATION: Config 5: CMN + RASTA + top_k=3
Optimized clean thresholds: theta_abs = 9.25, theta_margin = 1.30
Clean results: TAR = 61.5%, FAR = 2.6%, URR = 97.4%, Latency = 500.3ms


In [12]:
# Construct and display the side-by-side performance table
scenarios = [
    ("Clean (0dB Noise)", "Clean (0dB Noise)"),
    ("20dB Environmental Noise", "20dB Env"),
    ("20dB Speech Babble Noise", "20dB Babble"),
    ("10dB Environmental Noise", "10dB Env"),
    ("10dB Speech Babble Noise", "10dB Babble"),
    ("5dB Environmental Noise", "5dB Env"),
    ("5dB Speech Babble Noise", "5dB Babble")
]

comparison_rows = []
for label, short_label in scenarios:
    c1_tar, c1_far, c1_urr = cfg1_res[label]
    c2_tar, c2_far, c2_urr = cfg2_res[label]
    c3_tar, c3_far, c3_urr = cfg3_res[label]
    c4_tar, c4_far, c4_urr = cfg4_res[label]
    c5_tar, c5_far, c5_urr = cfg5_res[label]
    
    comparison_rows.append([
        short_label,
        f"{c1_tar*100:.1f}% / {c1_far*100:.1f}%",
        f"{c2_tar*100:.1f}% / {c2_far*100:.1f}%",
        f"{c3_tar*100:.1f}% / {c3_far*100:.1f}%",
        f"{c4_tar*100:.1f}% / {c4_far*100:.1f}%",
        f"{c5_tar*100:.1f}% / {c5_far*100:.1f}%"
    ])

print("\n" + "=" * 130)
print("                              VOXA V3 ROBUSTNESS COMPARISON REPORT")
print("                                   (Values: TAR% / FAR%)")
print("=" * 130)
headers = ["Scenario", "Config 1 (Baseline)", "Config 2 (Banded+AGC)", "Config 3 (CMN+Fine)", "Config 4 (CMN+FeatWarp)", "Config 5 (CMN+RASTA)"]
print(tabulate(comparison_rows, headers=headers, tablefmt="grid"))

# Latency Comparison
latency_rows = [
    ["Config 1 (Baseline)", f"{cfg1_lat:.2f} ms", "No (band_width=None)", "top_k=2"],
    ["Config 2 (Banded+AGC)", f"{cfg2_lat:.2f} ms", "Yes (Adaptive)", "top_k=2"],
    ["Config 3 (CMN+Fine)", f"{cfg3_lat:.2f} ms", "Yes (Adaptive)", "top_k=3"],
    ["Config 4 (CMN+FeatWarp)", f"{cfg4_lat:.2f} ms", "Yes (Adaptive)", "top_k=3"],
    ["Config 5 (CMN+RASTA)", f"{cfg5_lat:.2f} ms", "Yes (Adaptive)", "top_k=3"]
]
print("\n" + "=" * 90)
print("                         AVERAGE LATENCY COMPARISON")
print("=" * 90)
lat_headers = ["Configuration", "Average Latency (ms)", "Banded DTW Enabled", "Consensus top_k"]
print(tabulate(latency_rows, headers=lat_headers, tablefmt="grid"))


                              VOXA V3 ROBUSTNESS COMPARISON REPORT
                                   (Values: TAR% / FAR%)
+-------------------+-----------------------+-------------------------+-----------------------+---------------------------+------------------------+
| Scenario          | Config 1 (Baseline)   | Config 2 (Banded+AGC)   | Config 3 (CMN+Fine)   | Config 4 (CMN+FeatWarp)   | Config 5 (CMN+RASTA)   |
+===================+=======================+=========================+=======================+===========================+========================+
| Clean (0dB Noise) | 80.7% / 2.9%          | 81.3% / 2.9%            | 59.4% / 2.6%          | 75.4% / 2.6%              | 61.5% / 2.6%           |
+-------------------+-----------------------+-------------------------+-----------------------+---------------------------+------------------------+
| 20dB Env          | 62.0% / 1.1%          | 63.1% / 1.3%            | 4.8% / 1.8%           | 54.5% / 0.8%              | 8.6% /